# LSTM Model for Text Classification

This notebook handles the entire pipeline for training an LSTM model on text data:
1. Data loading and preprocessing
2. Text tokenization and sequence creation
3. Word embeddings
4. Hyperparameter tuning
5. Model training and evaluation
6. Model saving


In [17]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import re
import nltk
import os

nltk.download('punkt', quiet=True)

np.random.seed(2025)
tf.random.set_seed(2025)


## Data Preparation

In [18]:
print("Loading dataset...")
df = pd.read_csv("../datasets/custom_dataset.csv", sep="\t")
print(f"Dataset shape: {df.shape}")
df.head()


Loading dataset...
Dataset shape: (5437, 2)


,Text,Label
0,"In mechanics, a variable-mass system is a coll...",Human
1,Variable-mass systems in fluids involve object...,AI
2,"Geomechanics (from the Greek γεός, i.e. prefix...",Human
3,Geomechanics studies the mechanical behavior o...,AI
4,Microscale chemistry (often referred to as sma...,Human


## Predict Text With LSTM

In [19]:
import pickle

def predict_text(text, model, preprocessor):
    cleaned_text = preprocessor['clean_text'](text)
    
    sequence = preprocessor['tokenizer'].texts_to_sequences([cleaned_text])
    padded = pad_sequences(sequence, maxlen=preprocessor['max_seq_length'], padding='post', truncating='post')
    
    prediction = model.predict(padded)[0][0]
    
    return {
        'probability': float(prediction),
        'prediction': 'AI' if prediction > 0.5 else 'Human'
    }

loaded_model = keras.models.load_model('../trained_models/tensorflow/lstm_model.h5')
with open('../trained_models/tensorflow/lstm_tokenizer.pkl', 'rb') as f:
    loaded_preprocessor = pickle.load(f)

sample_text = "This is a sample text to test the model."
result = predict_text(sample_text, loaded_model, loaded_preprocessor)
print(f"Sample text: '{sample_text}'")
print(f"Prediction: {result['prediction']}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 942ms/step
Sample text: 'This is a sample text to test the model.'
Prediction: Human


## Load Dataset

In [20]:
print("Loading dataset...")
try:
    df = pd.read_csv('../datasets/submission3_inputs.csv', sep=';')
except:
    df = pd.read_csv('../datasets/submission3_inputs.csv')

print(f"Dataset loaded with {len(df)} entries")

df.head()

Loading dataset...
Dataset loaded with 100 entries


,ID,Text
0,D3-1,String theory is a broad and varied subject th...
1,D3-2,String theory is a theoretical framework in ph...
2,D3-3,String theory proposes that the fundamental bu...
3,D3-4,I think string theory explains only the 3rd di...
4,D3-5,"With all this said, one should keep in mind th..."


## Make Prediction with LSTM

In [21]:

print("Making predictions with LSTM model...")
lstm_predictions = []
for idx, row in df.iterrows():
    text = row['Text']
    prediction = predict_text(text, loaded_model, loaded_preprocessor)['prediction']
    lstm_predictions.append(prediction)
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(df)} entries with LSTM model")

lstm_results = pd.DataFrame({
    'ID': df['ID'],
    'Label': lstm_predictions
})

lstm_results.head()

Making predictions with LSTM model...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Processed 10/100 entries with LSTM model
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Processed 20/100 entries with LSTM model
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
1/1

,ID,Label
0,D3-1,Human
1,D3-2,Human
2,D3-3,AI
3,D3-4,Human
4,D3-5,Human


## Save Results

In [22]:

if not os.path.exists('results'):
    os.makedirs('results')

lstm_results.to_csv('results/submissao3-grupo011-s2.csv', sep='\t', index=False)